# HDD 15 — Five-model label-blind unsupervised detector pilot

**Purpose:** Train one interpretable, non-neural Isolation Forest per Notebook 13 model using 30-day SMART summaries; fit model-specific alert thresholds on separate calibration drives; score separate evaluation drives. Compare the resulting *development* warning evidence with the frozen Notebook 14 SMART baselines. This is a formal unsupervised baseline and a data/scoring workflow for the later masked Transformer study. It is **not** the previous single-model encoder, a temporal regime detector, or a blind Q4 result.

Run after Notebooks 13 and 14 in the same Python environment. July 1 drive IDs are assigned to fit/calibration/evaluation by deterministic hash before looking at their later readings. Feature extraction uses no failure flag. September recorded failures are joined only after the score manifests and thresholds are saved. Q4 and June remain closed. Since Notebook 14 already reported Q3 failure summaries, this pilot is exploratory; lock a final method before Q4.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib, json, re, zipfile
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score
try:
    from IPython.display import display
except ImportError:
    display=print

COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
BASE_ID='q3_multimodel_warning_baselines_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
BASE=COMPONENT_ROOT/'results'/RUN_ID/BASE_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/PILOT_ID
PARTS=OUT/'feature_partitions';MODEL_OUT=COMPONENT_ROOT/'models/trained'/RUN_ID/PILOT_ID
for p in [OUT,PARTS,MODEL_OUT]:p.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def load_json(path):
    assert path.is_file(),f'Missing input: {path}'
    return json.loads(path.read_text())
audit=load_json(AUDIT/'multimodel_coverage_complete.json')
base=load_json(BASE/'baseline_evaluation_complete.json')
assert audit['analysis_complete'] and base['analysis_complete']
assert not audit['future_test_outcomes_opened'] and not base['future_test_outcomes_opened']
assert not audit['sealed_June_test_opened'] and not base['sealed_June_test_opened']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==audit['split_sha256']
reservation=load_json(reservation_path)
assert sha256(Q3_ZIP)==audit['settings']['archive_sha256']
models=list(reservation['models']);assert len(models)==5
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
SALT='five-model-if-pilot-v1-seed42'
def group_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
groups={str(s):(m,group_for(s)) for m,v in reservation['models'].items() for s in v['development']}
assert set(groups).isdisjoint(reserved)
counts=pd.Series([m+' / '+g for m,g in groups.values()]).value_counts().sort_index()
display(counts)
assert all(sum(1 for m,g in groups.values() if m==name and g==role)>100 for name in models for role in ['fit','calibration','evaluation'])
FEATURES=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
F_NAMES=['last_'+f for f in FEATURES]+['std30_'+f for f in FEATURES]+['positive7_'+f for f in ['smart_5_raw','smart_197_raw','smart_198_raw']]
DATES=pd.date_range('2021-07-01','2021-09-30')
DAY={str(x.date()):i for i,x in enumerate(DATES)}
SETTINGS={'salt':SALT,'split_cutoffs':[.60,.80],'fit_snapshot_date':'2021-08-15',
          'calibration_dates':['2021-08-16','2021-08-31'],
          'evaluation_score_dates':['2021-08-01','2021-09-30'],
          'evaluation_metric_dates':['2021-09-01','2021-09-30'],
          'warning_horizons':[7,14,30],'window_days':30,
          'n_estimators':100,'max_samples':256,'seed':42,
          'calibration_quantile':.998,'strict_alarm':'score > threshold',
          'representation':'log1p last five values, log1p 30-day std five values, log1p positive 7-day sector changes',
          'reservation_sha256':sha256(reservation_path),'q3_sha256':sha256(Q3_ZIP),
          'upstream_baseline_complete_sha256':sha256(BASE/'baseline_evaluation_complete.json')}
settings_path=OUT/'settings.json'
if settings_path.exists():assert load_json(settings_path)==SETTINGS,'Settings changed: use a new PILOT_ID.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
print('No Q4 observations or failure flags loaded. Model groups fixed by July 1 IDs.')

HGST HMS5C4040BLE640 / calibration     1982
HGST HMS5C4040BLE640 / evaluation      2028
HGST HMS5C4040BLE640 / fit             6115
ST12000NM0008 / calibration            2949
ST12000NM0008 / evaluation             3063
ST12000NM0008 / fit                    9094
ST4000DM000 / calibration              2937
ST4000DM000 / evaluation               3055
ST4000DM000 / fit                      9042
ST8000NM0055 / calibration             2343
ST8000NM0055 / evaluation              2270
ST8000NM0055 / fit                     6908
TOSHIBA MG07ACA14TA / calibration      5067
TOSHIBA MG07ACA14TA / evaluation       5173
TOSHIBA MG07ACA14TA / fit             15386
Name: count, dtype: int64

No Q4 observations or failure flags loaded. Model groups fixed by July 1 IDs.


## 1. Extract label-blind 30-day features

A missing or invalid day resets the window. The fit set takes one snapshot per drive on August 15, preventing overlapping windows from giving some drives extra weight. Calibration uses different drives on August 16–31. Evaluation-drive scores are prepared from August 1 for September event histories. The feature partitions are reused on reruns if their manifest matches the frozen input identity.

In [2]:
def featurize(window):
    a=np.asarray(window,dtype=np.float64)
    assert a.shape==(30,5)
    last=np.log1p(a[-1])
    std=np.log1p(a.std(axis=0))
    positive7=np.log1p(np.maximum(a[-1,[0,3,4]]-a[-8,[0,3,4]],0))
    return np.concatenate([last,std,positive7]).astype(np.float32)

with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DAY)

manifest_path=OUT/'feature_manifest.json'
identity=sha256(settings_path)
if manifest_path.exists():
    feature_manifest=load_json(manifest_path)
    assert feature_manifest['settings_sha256']==identity
    for file,digest in feature_manifest['partition_sha256'].items():
        assert sha256(PARTS/file)==digest,'Feature partition changed.'
    assert len(feature_manifest['partition_sha256'])==61
    print('Verified cached label-blind feature partitions:',len(feature_manifest['partition_sha256']))
else:
    # Keep only the last 30 valid consecutive values per declared development drive.
    history={};last_seen={};observed_bits={};valid_bits={};partition_sha={};quality={m:0 for m in models}
    for date,member in sorted(members.items()):
        day=DAY[date];seen=set();rows=[]
        with zipfile.ZipFile(Q3_ZIP) as z,z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number']+FEATURES,dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(groups)]
                if part.empty:continue
                assert part.date.astype(str).eq(date).all()
                serials=part.serial_number.tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials)
                seen.update(serials)
                values=part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
                for serial,v in zip(serials,values):
                    observed_bits[serial]=observed_bits.get(serial,0)|(1<<day)
                    valid=bool(np.isfinite(v).all() and (v>=0).all())
                    if not valid:
                        history.pop(serial,None);last_seen[serial]=day;continue
                    if last_seen.get(serial,-2)!=day-1:history[serial]=deque(maxlen=30)
                    h=history.setdefault(serial,deque(maxlen=30));h.append(v.astype(np.float32));last_seen[serial]=day
                    if len(h)<30:continue
                    valid_bits[serial]=valid_bits.get(serial,0)|(1<<day)
                    model,role=groups[serial];quality[model]+=1
                    take=(role=='fit' and date=='2021-08-15') or (role=='calibration' and '2021-08-16'<=date<='2021-08-31') or (role=='evaluation' and date>='2021-08-01')
                    if take:rows.append((serial,model,role,date,*featurize(h)))
        if date>='2021-08-01':
            frame=pd.DataFrame(rows,columns=['serial_number','model','role','end_date']+F_NAMES)
            path=PARTS/f'{date}.parquet';frame.to_parquet(path,index=False)
            partition_sha[path.name]=sha256(path)
        if date.endswith(('-01','-15')):print('Extracted:',date)
    # Notebook 13 independently counted every valid 30-day window for this exact population.
    earlier=pd.read_csv(AUDIT/'model_event_coverage_summary.csv').set_index('model')
    for model in models:assert quality[model]==int(earlier.loc[model,'quality_30day_windows']),(model,quality[model])
    feature_manifest={'settings_sha256':identity,'partition_sha256':partition_sha,
                      'quality_windows':quality,'reserved_drives_excluded':len(reserved)}
    manifest_path.write_text(json.dumps(feature_manifest,indent=2))
    print('Notebook 13 quality counts reproduced:',sum(quality.values()))
print('Feature extraction complete. Failure labels have not been read in this notebook.')

Extracted: 2021-07-01
Extracted: 2021-07-15
Extracted: 2021-08-01
Extracted: 2021-08-15
Extracted: 2021-09-01
Extracted: 2021-09-15
Notebook 13 quality counts reproduced: 4848446
Feature extraction complete. Failure labels have not been read in this notebook.


## 2. Train and calibrate five independent Isolation Forests

All preprocessing and model fits use only the fit-drive snapshots. Model-specific thresholds use only calibration drives and the predeclared 99.8th percentile. `-score_samples` is larger for more unusual windows. No failure flag is used. The cached model bundle is tied to the feature manifest and settings; a changed input raises an error.

In [3]:
fit_frame=pd.read_parquet(PARTS/'2021-08-15.parquet')
fit_frame=fit_frame.loc[fit_frame.role.eq('fit')].copy()
cal_frames=[pd.read_parquet(PARTS/f'{d}.parquet') for d in DAY if '2021-08-16'<=d<='2021-08-31']
cal_frame=pd.concat(cal_frames,ignore_index=True)
cal_frame=cal_frame.loc[cal_frame.role.eq('calibration')].copy()
assert not fit_frame.duplicated(['serial_number']).any()
assert set(fit_frame.role)=={'fit'} and set(cal_frame.role)=={'calibration'}
assert set(fit_frame.serial_number).isdisjoint(cal_frame.serial_number)
artifact_path=MODEL_OUT/'if_models.joblib';spec_path=MODEL_OUT/'if_specification.json'
training_identity=hashlib.sha256((sha256(settings_path)+sha256(manifest_path)).encode()).hexdigest()
if artifact_path.exists() and spec_path.exists():
    model_spec=load_json(spec_path)
    assert model_spec['training_identity']==training_identity and sha256(artifact_path)==model_spec['artifact_sha256']
    fitted=joblib.load(artifact_path) # Load only the artifact you created locally.
    print('Verified cached five-model detector.')
else:
    fitted={};meta={}
    for model in models:
        fit=fit_frame.loc[fit_frame.model.eq(model),F_NAMES].to_numpy(dtype=np.float32)
        cal=cal_frame.loc[cal_frame.model.eq(model),F_NAMES].to_numpy(dtype=np.float32)
        assert len(fit)>100 and len(cal)>100 and np.isfinite(fit).all() and np.isfinite(cal).all()
        scaler=RobustScaler().fit(fit)
        detector=IsolationForest(n_estimators=SETTINGS['n_estimators'],max_samples=SETTINGS['max_samples'],
                                 random_state=SETTINGS['seed'],n_jobs=-1).fit(scaler.transform(fit))
        cal_scores=-detector.score_samples(scaler.transform(cal))
        threshold=float(np.quantile(cal_scores,SETTINGS['calibration_quantile'],method='higher'))
        fitted[model]={'scaler':scaler,'detector':detector,'threshold':threshold}
        meta[model]={'fit_drives':len(fit),'calibration_windows':len(cal),
                     'calibration_drives':int(cal_frame.loc[cal_frame.model.eq(model),'serial_number'].nunique()),
                     'threshold':threshold}
    joblib.dump(fitted,artifact_path)
    model_spec={'training_identity':training_identity,'artifact_sha256':sha256(artifact_path),
                'thresholds':meta,'failure_labels_used':False,'future_test_opened':False}
    spec_path.write_text(json.dumps(model_spec,indent=2))
display(pd.DataFrame(model_spec['thresholds']).T)
assert set(fitted)==set(models)

,fit_drives,calibration_windows,calibration_drives,threshold
TOSHIBA MG07ACA14TA,15374.0,80930.0,5059.0,0.619010
ST12000NM0008,9044.0,46916.0,2938.0,0.650098
ST4000DM000,9001.0,46666.0,2918.0,0.751943
ST8000NM0055,6864.0,37263.0,2333.0,0.732652
HGST HMS5C4040BLE640,6089.0,31624.0,1981.0,0.620698


## 3. Score evaluation drives and freeze score partitions

All eligible evaluation-drive endpoints from August 1 to September 30 are scored at thresholds fixed above. August scores supply complete histories for September events. Saved scores retain exact drive/date identities. No failure data is consulted until all score partitions and hashes are frozen.

In [4]:
SCORE_PARTS=OUT/'score_partitions';SCORE_PARTS.mkdir(exist_ok=True)
score_hashes={}
for date in DAY:
    if date<'2021-08-01':continue
    feature=pd.read_parquet(PARTS/f'{date}.parquet')
    part=feature.loc[feature.role.eq('evaluation'),['serial_number','model','end_date']+F_NAMES].copy()
    scored=[]
    for model in models:
        sub=part.loc[part.model.eq(model)]
        if sub.empty:continue
        bundle=fitted[model]
        x=bundle['scaler'].transform(sub[F_NAMES].to_numpy(dtype=np.float32))
        scores=-bundle['detector'].score_samples(x)
        scored.append(pd.DataFrame({'serial_number':sub.serial_number.to_numpy(),
                                    'model':model,'end_date':date,'score':scores,
                                    'threshold':bundle['threshold'],
                                    'is_anomaly_candidate':scores>bundle['threshold']}))
    table=pd.concat(scored,ignore_index=True) if scored else pd.DataFrame(columns=['serial_number','model','end_date','score','threshold','is_anomaly_candidate'])
    assert not table.duplicated(['serial_number','end_date']).any()
    assert set(table.serial_number).isdisjoint(reserved)
    path=SCORE_PARTS/f'{date}.parquet'
    if path.exists():assert pd.read_parquet(path).equals(table),f'Score partition changed: {path}'
    else:table.to_parquet(path,index=False)
    score_hashes[path.name]=sha256(path)
    if date.endswith(('-01','-15')):print('Scored:',date)
score_manifest={'settings_sha256':sha256(settings_path),'feature_manifest_sha256':sha256(manifest_path),
                'model_specification_sha256':sha256(spec_path),'score_sha256':score_hashes,
                'failure_labels_used_for_scoring':False,'future_test_opened':False}
score_manifest_path=OUT/'score_manifest.json'
if score_manifest_path.exists():assert load_json(score_manifest_path)==score_manifest,'Score manifest changed.'
else:score_manifest_path.write_text(json.dumps(score_manifest,indent=2))
print('Score partitions frozen before failure-event evaluation:',len(score_hashes))

Scored: 2021-08-01
Scored: 2021-08-15
Scored: 2021-09-01
Scored: 2021-09-15
Score partitions frozen before failure-event evaluation: 61


## 4. Join first failures only for development evaluation

A positive endpoint precedes its first recorded failure by 1–h days. A negative endpoint needs every later day observed through h, with no earlier recorded failure. The September evaluation drive is held out from fitting and calibration, but Q3 aggregate labels were already reviewed in Notebook 14, so treat this as exploratory. False-alert denominator is *observed negative endpoints*, not fleet-wide operating drive-days. September events with partial pre-failure score history are excluded from strict recall.

In [5]:
event_path=COMPONENT_ROOT/'data/evaluation_only'/RUN_ID/AUDIT_ID/'development_first_failure_support.csv'
events=pd.read_csv(event_path,dtype={'serial_number':str},parse_dates=['failure_date'])
events=events.loc[events.serial_number.map(lambda s:groups.get(s,(None,None))[1]).eq('evaluation')].copy()
assert not events.serial_number.duplicated().any()
assert set(events.serial_number).isdisjoint(reserved)
first={s:DAY[str(pd.Timestamp(d).date())] for s,d in events[['serial_number','failure_date']].itertuples(index=False,name=None)}
# Use label-free Notebook 14 feature availability to reconstruct observation-day bit masks.
# The current run also supports cache-only execution without reopening failure flags.
obs={};score_valid={}
for date in DAY:
    if date<'2021-08-01':continue
    q=pd.read_parquet(SCORE_PARTS/f'{date}.parquet',columns=['serial_number'])
    bit=1<<DAY[date]
    for serial in q.serial_number:score_valid[serial]=score_valid.get(serial,0)|bit
# For negative follow-up, read serial/date only. Never load reserved-drive observations.
with zipfile.ZipFile(Q3_ZIP) as z:
    for date,member in sorted(members.items()):
        bit=1<<DAY[date];seen=set()
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['serial_number'],dtype={'serial_number':str},chunksize=50000):
                serials=chunk.loc[chunk.serial_number.map(lambda s:groups.get(s,(None,None))[1]).eq('evaluation'),'serial_number'].tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials)
                seen.update(serials)
                for serial in serials:obs[serial]=obs.get(serial,0)|bit

def complete(bits,lo,hi):
    if lo<0 or hi>=len(DATES):return False
    return bits&(((1<<(hi-lo+1))-1)<<lo)==(((1<<(hi-lo+1))-1)<<lo)

scores=pd.concat([pd.read_parquet(SCORE_PARTS/f'{date}.parquet') for date in DAY if date>='2021-08-01'],ignore_index=True)
assert not scores.duplicated(['serial_number','end_date']).any()
assert np.isfinite(scores[['score','threshold']].to_numpy()).all()
assert (scores.is_anomaly_candidate.to_numpy()==(scores.score>scores.threshold).to_numpy()).all()
results=[];details=[]
for model in models:
    all_model=scores.loc[scores.model.eq(model)]
    september=all_model.loc[all_model.end_date.ge('2021-09-01')]
    event_model=events.loc[events.model.eq(model)]
    for h in SETTINGS['warning_horizons']:
        y=[];negative_mask=[]
        for serial,date in september[['serial_number','end_date']].itertuples(index=False,name=None):
            d=DAY[date];failure=first.get(serial)
            if failure is not None and failure<=d:target=np.nan
            elif failure is not None and 0<failure-d<=h:target=1
            elif complete(obs.get(serial,0),d+1,d+h):target=0
            else:target=np.nan
            y.append(target)
        target=np.asarray(y,dtype=float);keep=np.isfinite(target)
        v=target[keep].astype(int);s=september.score.to_numpy()[keep];f=september.is_anomaly_candidate.to_numpy()[keep]
        pos=int((v==1).sum());neg=int((v==0).sum());tp=int((f&(v==1)).sum());fp=int((f&(v==0)).sum())
        drows=[]
        for event in event_model.itertuples(index=False):
            fail=pd.Timestamp(event.failure_date)
            if not pd.Timestamp('2021-09-01')<=fail<=pd.Timestamp('2021-09-30'):continue
            d=DAY[str(fail.date())]
            history=all_model.loc[all_model.serial_number.eq(event.serial_number)&all_model.end_date.isin([str(DATES[x].date()) for x in range(max(0,d-h),d)])]
            full=bool(len(history)==h and complete(score_valid.get(event.serial_number,0),d-h,d-1))
            alerted=history.loc[history.is_anomaly_candidate]
            lead=int((fail-pd.Timestamp(alerted.end_date.min())).days) if full and len(alerted) else np.nan
            drows.append({'model':model,'horizon_days':h,'serial_number':event.serial_number,
                          'failure_date':fail,'complete_warning_history':full,
                          'recalled':bool(full and len(alerted)),'first_warning_lead_days':lead})
        ed=pd.DataFrame(drows,columns=['model','horizon_days','serial_number','failure_date','complete_warning_history','recalled','first_warning_lead_days'])
        full=ed.loc[ed.complete_warning_history];rec=full.loc[full.recalled]
        details.append(ed)
        results.append({'model':model,'horizon_days':h,'evaluation_drives':int(september.serial_number.nunique()),
                        'scored_endpoints':len(september),'positive_endpoints':pos,'negative_endpoints':neg,
                        'censored_or_post_failure':int((~keep).sum()),
                        'average_precision':float(average_precision_score(v,s)) if pos and neg else np.nan,
                        'positive_prevalence':float(v.mean()) if len(v) else np.nan,
                        'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
                        'positive_endpoint_recall':tp/pos if pos else np.nan,
                        'false_alerts_per_1000_observed_negative_endpoints':1000*fp/neg if neg else np.nan,
                        'all_scored_alerts_per_1000_endpoints':1000*september.is_anomaly_candidate.mean(),
                        'September_recorded_events':len(ed),'events_with_complete_warning_history':len(full),
                        'events_recalled':len(rec),'strict_event_recall':len(rec)/len(full) if len(full) else np.nan,
                        'median_first_warning_lead_days':float(rec.first_warning_lead_days.median()) if len(rec) else np.nan,
                        'status':'both_endpoint_classes' if pos and neg else 'insufficient_endpoint_classes'})
metrics=pd.DataFrame(results)
metrics.to_csv(OUT/'isolation_forest_development_metrics.csv',index=False)
pd.concat(details,ignore_index=True).to_csv(OUT/'isolation_forest_event_details.csv',index=False)
display(metrics[['model','horizon_days','positive_endpoints','average_precision','events_with_complete_warning_history','strict_event_recall','false_alerts_per_1000_observed_negative_endpoints']])
print('Do not compare these metrics directly with Notebook 14: the drive sets and September endpoint range differ.')

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_23548\250279594.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(details,ignore_index=True).to_csv(OUT/'isolation_forest_event_details.csv',index=False)


,model,horizon_days,positive_endpoints,average_precision,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints
0,TOSHIBA MG07ACA14TA,7,14,0.000079,2,0.000000,0.968470
1,TOSHIBA MG07ACA14TA,14,28,0.000229,2,0.000000,1.053307
2,TOSHIBA MG07ACA14TA,30,35,NaN,2,0.000000,NaN
3,ST12000NM0008,7,28,0.054345,4,0.500000,1.828075
4,ST12000NM0008,14,53,0.162146,4,0.500000,1.744270
5,ST12000NM0008,30,80,NaN,4,0.500000,NaN
6,ST4000DM000,7,49,0.044482,7,0.428571,2.251251
7,ST4000DM000,14,92,0.058691,7,0.428571,2.105220
8,ST4000DM000,30,124,NaN,7,0.428571,NaN
9,ST8000NM0055,7,21,0.007227,4,0.250000,1.954690


Do not compare these metrics directly with Notebook 14: the drive sets and September endpoint range differ.


## 5. Record the comparison boundary and next experiment

Notebook 14’s fixed SMART results use *all* 77,412 declared development drives and August 16–September 30 endpoints. This pilot uses a held-out fifth of those drives and September endpoints. A fair comparison must rescore the fixed SMART baselines on these exact evaluation drives/dates and compare at the same alert budget. The masked Transformer needs its own five-model training and frozen scores before its failure comparison.

In [6]:
assert sha256(reservation_path)==SETTINGS['reservation_sha256']
assert sha256(Q3_ZIP)==SETTINGS['q3_sha256']
assert {p.name:sha256(p) for p in SCORE_PARTS.glob('*.parquet')}==score_hashes
assert len(metrics)==len(models)*3
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'five_models':models,'group_counts':counts.to_dict(),
        'settings_sha256':sha256(settings_path),'feature_manifest_sha256':sha256(manifest_path),
        'model_spec_sha256':sha256(spec_path),'score_manifest_sha256':sha256(score_manifest_path),
        'metric_sha256':sha256(OUT/'isolation_forest_development_metrics.csv'),
        'failure_labels_used_for_training_or_calibration':False,
        'failure_labels_used_for_evaluation':True,'future_test_outcomes_opened':False,
        'sealed_June_test_opened':False,'masked_transformer_trained_on_five_models':False,
        'limitations':['Q3 labels were already seen in Notebook 14; exploratory development only',
                       'Notebook 14 baselines use a different drive/date population; compare only after matched subset evaluation',
                       'one snapshot per drive for fit; single seed and no uncertainty intervals',
                       'device-type verification pending for four models','HGST event support may be insufficient']}
(OUT/'pilot_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 15 COMPLETE — five-model label-blind Isolation Forest pilot saved.')
print('Q4 outcomes unopened. Results:',OUT)

NOTEBOOK 15 COMPLETE — five-model label-blind Isolation Forest pilot saved.
Q4 outcomes unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_five_model_isolation_forest_v1
